### 3.3.30. Convex Programming Problem Classes

$$
\mathrm{LP}\subseteq\mathrm{QP}\subseteq\mathrm{QCQP}\subseteq\mathrm{SOCP}\subseteq\mathrm{SDP}\subseteq\mathrm{CP}
$$


**Explanation:**

Convex programming classes are organized by the structure allowed in the objective and constraints. Linear programs use linear objectives and affine constraints; quadratic programs add a positive semidefinite quadratic objective; QCQPs add convex quadratic constraints; SOCPs use second-order cone constraints; SDPs use positive semidefinite matrix constraints; and cone programs express feasibility through membership in a convex cone.

The hierarchy means that a more general class can represent many narrower classes, although the narrowest matching formulation is usually the clearest model and often the better computational target. This ordering connects directly to the later notebooks on [linear programming](05_linear_programming_forms_and_examples.ipynb), [quadratic programming and QCQP](06_quadratic_programming_and_qcqp.ipynb), [second-order cone programming](07_second_order_cone_programming.ipynb), [semidefinite programming](08_semidefinite_programming_and_lmis.ipynb), and [conic form](09_conic_form_optimization.ipynb).

**Properties:**
- LP: linear objective with affine equality and inequality constraints.
- QP: convex quadratic objective with affine constraints.
- QCQP: convex quadratic objective with convex quadratic constraints.
- SOCP: linear objective with second-order cone constraints.
- SDP: linear objective with affine positive semidefinite matrix constraints.
- CP: linear objective with membership in a general convex cone.

**Intuition:**

The hierarchy figure shows the containment picture emphasized in the Tordesillas transcript: LPs are the most specialized class in this list, while broader classes can represent narrower ones. The class-form figure lists the algebraic templates: QPs add a positive semidefinite quadratic objective, QCQPs add positive semidefinite quadratic constraints, SOCPs add second-order cone constraints, SDPs add positive semidefinite matrix constraints, and cone programs use a general convex cone. The examples figure connects those templates to geometry: polyhedra give LP feasible sets, ellipsoids and paraboloids give QCQP structure, second-order cones give SOCP constraints, and positive semidefinite cones give SDP constraints.

<img src="../../Figures/030304_tordesillas_convex_programming_problem_classes.png" alt="Tordesillas convex programming problem-class containment hierarchy" style="width: 55%; height: auto;">

<img src="../../Figures/030304_tordesillas_convex_programming_class_forms.png" alt="Tordesillas screenshot of LP, QP, QCQP, SOCP, SDP, and cone program forms" style="width: 85%; height: auto;">

<img src="../../Figures/030304_tordesillas_convex_programming_class_examples.png" alt="Tordesillas screenshot of geometric examples of convex programming classes" style="width: 85%; height: auto;">


**Numerical Example:**

The examples figure labels a quadratic objective over affine constraints as a QP. Consider
$$
\begin{aligned}
\min_{x_1,x_2}\quad & \frac12
\begin{bmatrix}x_1&x_2\end{bmatrix}
\begin{bmatrix}2&0\\0&0\end{bmatrix}
\begin{bmatrix}x_1\\x_2\end{bmatrix}
+
\begin{bmatrix}0&1\end{bmatrix}
\begin{bmatrix}x_1\\x_2\end{bmatrix} \\
\text{subject to}\quad & x_1+x_2=1,\quad x_1\geq0,\quad x_2\geq0.
\end{aligned}
$$

The quadratic matrix is
$$
P=\begin{bmatrix}2&0\\0&0\end{bmatrix},
$$
whose eigenvalues are $2$ and $0$. Therefore $P\succeq0$.

The objective expands to
$$
\frac12(2x_1^2)+x_2=x_1^2+x_2,
$$
and the constraints are affine. The problem is therefore a convex QP.

Using $x_2=1-x_1$ gives
$$
f(x_1)=x_1^2+1-x_1.
$$
The unconstrained stationary point satisfies
$$
f'(x_1)=2x_1-1=0,
$$
so
$$
x_1^\star=\frac12,\qquad x_2^\star=\frac12.
$$
The minimum value is
$$
f\left(\frac12\right)=\frac14+\frac12=\frac34.
$$


In [ ]:
import sympy as sp

x_1, x_2 = sp.symbols("x_1 x_2")
x = sp.Matrix([x_1, x_2])
P = sp.Matrix([[2, 0], [0, 0]])
q = sp.Matrix([0, 1])
objective = sp.Rational(1, 2) * (x.T * P * x)[0] + (q.T * x)[0]
simplex_objective = sp.expand(objective.subs(x_2, 1 - x_1))
stationary_solution = sp.solve(sp.diff(simplex_objective, x_1), x_1)[0]
optimizer = sp.Matrix([stationary_solution, 1 - stationary_solution])
minimum_value = sp.simplify(simplex_objective.subs(x_1, stationary_solution))

print("objective ="); sp.pprint(objective)
print("P eigenvalues =", P.eigenvals())
print("objective on simplex =", simplex_objective)
print("optimizer ="); sp.pprint(optimizer)
print("minimum_value =", minimum_value)


**Equivalent `casadi` implementation:**


In [ ]:
import casadi as ca

opti = ca.Opti()
x = opti.variable(2)
P = ca.DM([[2, 0], [0, 0]])
q = ca.DM([0, 1])
objective = 0.5 * ca.bilin(P, x, x) + ca.dot(q, x)

opti.minimize(objective)
opti.subject_to(x[0] + x[1] == 1)
opti.subject_to(x >= 0)
opti.set_initial(x, ca.DM([0.5, 0.5]))
opti.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes"})
solution = opti.solve()

print("optimizer =", ca.DM(solution.value(x)))
print("minimum_value =", ca.DM(solution.value(opti.f)))


**Canonical Program Forms:**

The six classes differ by the objective curvature and by the allowed constraint set:

$$
\begin{array}{lll}
\mathrm{LP} & \min_x c^Tx & A_1x\leq b_1,\ A_2x=b_2 \\
\mathrm{QP} & \min_x \frac12x^TPx+q^Tx+r & A_1x\leq b_1,\ A_2x=b_2,\ P\succeq0 \\
\mathrm{QCQP} & \min_x \frac12x^TPx+q^Tx+r & \frac12x^TP_ix+q_i^Tx+r_i\leq0,\ P_i\succeq0 \\
\mathrm{SOCP} & \min_x c^Tx & \|M_jx+s_j\|_2\leq c_j^Tx+d_j \\
\mathrm{SDP} & \min_x c^Tx & F_0+\sum_i x_iF_i\succeq0 \\
\mathrm{CP} & \min_x c^Tx & x\in K,\ K\text{ a convex cone.}
\end{array}
$$

Positive semidefinite matrices and convex cones are what allow the later classes to encode broader feasible regions while preserving convexity.


**References:**

[📗 Tordesillas, J. (n.d.). *Optimization, Optimal Control, Trajectory Optimization, and Splines*.](https://www.youtube.com/watch?v=j82Ia436DYY)  
[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)

---

[⬅️ Previous: Convex Problem Standard Form and Optimality](./29_convex_problem_standard_form_and_optimality.ipynb) | [Next: Quasiconvex Optimization via Feasibility ➡️](./31_quasiconvex_optimization_via_feasibility.ipynb)

---
